In [ ]:
# Load MSE data for all experiments
mse_cntl = xr.open_dataset(data_path / 'mse_cntl.nc')
mse_p4k = xr.open_dataset(data_path / 'mse_p4k.nc')
mse_4co2 = xr.open_dataset(data_path / 'mse_4co2.nc')

print("CNTL MSE data:")
print(mse_cntl)
print("\n" + "="*80 + "\n")
print("P4K MSE data:")
print(mse_p4k)
print("\n" + "="*80 + "\n")
print("4CO2 MSE data:")
print(mse_4co2)

## 1. Load MSE Data

Load MSE data for all three experiments

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Set data path
data_path = Path('/work/mh1498/m301257/processed_data')

# Calculate dh/dt from MSE Data

This notebook calculates the time tendency of moist static energy (dh/dt) and compares differences between experiments (CNTL, P4K, 4CO2).

## 2. Calculate dh/dt (Time Tendency of MSE)

Calculate the time derivative of MSE using centered finite differences

In [ ]:
def calculate_dhdt(ds, mse_var='mse'):
    """
    Calculate dh/dt (time tendency of MSE) using centered finite differences
    
    Parameters:
    -----------
    ds : xarray.Dataset
        Dataset containing MSE data
    mse_var : str
        Name of the MSE variable in the dataset
    
    Returns:
    --------
    dhdt : xarray.DataArray
        Time tendency of MSE
    """
    # Get MSE variable
    if mse_var in ds:
        mse = ds[mse_var]
    else:
        # Try to find MSE-like variables
        possible_vars = [v for v in ds.data_vars if 'mse' in v.lower() or 'h' in v.lower()]
        if possible_vars:
            mse = ds[possible_vars[0]]
            print(f"Using variable: {possible_vars[0]}")
        else:
            raise ValueError(f"Cannot find MSE variable in dataset. Available variables: {list(ds.data_vars)}")
    
    # Calculate time tendency using centered finite differences
    # dh/dt ≈ (h(t+1) - h(t-1)) / (2*dt)
    dhdt = mse.differentiate('time')
    
    return dhdt

# Calculate dh/dt for all experiments
dhdt_cntl = calculate_dhdt(mse_cntl)
dhdt_p4k = calculate_dhdt(mse_p4k)
dhdt_4co2 = calculate_dhdt(mse_4co2)

print("dh/dt calculated successfully!")
print(f"CNTL dh/dt shape: {dhdt_cntl.shape}")
print(f"P4K dh/dt shape: {dhdt_p4k.shape}")
print(f"4CO2 dh/dt shape: {dhdt_4co2.shape}")

## 3. Calculate Differences Between Experiments

In [ ]:
# Calculate differences (anomalies relative to control)
dhdt_p4k_diff = dhdt_p4k - dhdt_cntl
dhdt_4co2_diff = dhdt_4co2 - dhdt_cntl

# Calculate climatological means
dhdt_cntl_mean = dhdt_cntl.mean(dim='time')
dhdt_p4k_mean = dhdt_p4k.mean(dim='time')
dhdt_4co2_mean = dhdt_4co2.mean(dim='time')

dhdt_p4k_diff_mean = dhdt_p4k_diff.mean(dim='time')
dhdt_4co2_diff_mean = dhdt_4co2_diff.mean(dim='time')

print("Climatological mean dh/dt calculated!")
print(f"CNTL mean dh/dt: {dhdt_cntl_mean.values.mean():.2e} J/kg/s")
print(f"P4K mean dh/dt: {dhdt_p4k_mean.values.mean():.2e} J/kg/s")
print(f"4CO2 mean dh/dt: {dhdt_4co2_mean.values.mean():.2e} J/kg/s")
print(f"\nP4K - CNTL: {dhdt_p4k_diff_mean.values.mean():.2e} J/kg/s")
print(f"4CO2 - CNTL: {dhdt_4co2_diff_mean.values.mean():.2e} J/kg/s")

## 4. Visualize dh/dt Spatial Patterns

In [ ]:
# Plot climatological mean dh/dt for all experiments
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

# Plot climatological means
im1 = dhdt_cntl_mean.plot(ax=axes[0, 0], cmap='RdBu_r', robust=True, 
                          cbar_kwargs={'label': 'dh/dt (J/kg/s)'})
axes[0, 0].set_title('CNTL: Climatological Mean dh/dt')

im2 = dhdt_p4k_mean.plot(ax=axes[1, 0], cmap='RdBu_r', robust=True,
                         cbar_kwargs={'label': 'dh/dt (J/kg/s)'})
axes[1, 0].set_title('P4K: Climatological Mean dh/dt')

im3 = dhdt_4co2_mean.plot(ax=axes[2, 0], cmap='RdBu_r', robust=True,
                          cbar_kwargs={'label': 'dh/dt (J/kg/s)'})
axes[2, 0].set_title('4CO2: Climatological Mean dh/dt')

# Plot differences
im4 = dhdt_p4k_diff_mean.plot(ax=axes[1, 1], cmap='RdBu_r', robust=True,
                              cbar_kwargs={'label': 'Δ(dh/dt) (J/kg/s)'})
axes[1, 1].set_title('P4K - CNTL: dh/dt Difference')

im5 = dhdt_4co2_diff_mean.plot(ax=axes[2, 1], cmap='RdBu_r', robust=True,
                               cbar_kwargs={'label': 'Δ(dh/dt) (J/kg/s)'})
axes[2, 1].set_title('4CO2 - CNTL: dh/dt Difference')

# Remove unused subplot
axes[0, 1].axis('off')

plt.tight_layout()
plt.savefig('/work/mh1498/m301257/figures/dhdt_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Spatial patterns plotted!")

## 5. Statistical Analysis of dh/dt Differences

In [ ]:
# Calculate global and tropical statistics
def calc_statistics(data, lat_bounds=(-30, 30)):
    """
    Calculate global and tropical mean statistics
    
    Parameters:
    -----------
    data : xarray.DataArray
        Data to analyze
    lat_bounds : tuple
        Latitude bounds for tropical region
    
    Returns:
    --------
    dict : Statistics dictionary
    """
    # Get latitude coordinate name
    lat_name = 'lat' if 'lat' in data.dims else 'latitude'
    
    # Global mean
    global_mean = data.mean(dim=[lat_name, 'lon' if 'lon' in data.dims else 'longitude']).values
    
    # Tropical mean
    tropical_data = data.sel({lat_name: slice(lat_bounds[0], lat_bounds[1])})
    tropical_mean = tropical_data.mean(dim=[lat_name, 'lon' if 'lon' in data.dims else 'longitude']).values
    
    # Standard deviation
    global_std = data.std(dim=[lat_name, 'lon' if 'lon' in data.dims else 'longitude']).values
    
    return {
        'global_mean': global_mean,
        'tropical_mean': tropical_mean,
        'global_std': global_std
    }

# Calculate statistics for mean fields
stats_cntl = calc_statistics(dhdt_cntl_mean)
stats_p4k = calc_statistics(dhdt_p4k_mean)
stats_4co2 = calc_statistics(dhdt_4co2_mean)
stats_p4k_diff = calc_statistics(dhdt_p4k_diff_mean)
stats_4co2_diff = calc_statistics(dhdt_4co2_diff_mean)

# Create summary table
summary_df = pd.DataFrame({
    'Experiment': ['CNTL', 'P4K', '4CO2', 'P4K-CNTL', '4CO2-CNTL'],
    'Global Mean (J/kg/s)': [
        stats_cntl['global_mean'],
        stats_p4k['global_mean'],
        stats_4co2['global_mean'],
        stats_p4k_diff['global_mean'],
        stats_4co2_diff['global_mean']
    ],
    'Tropical Mean (J/kg/s)': [
        stats_cntl['tropical_mean'],
        stats_p4k['tropical_mean'],
        stats_4co2['tropical_mean'],
        stats_p4k_diff['tropical_mean'],
        stats_4co2_diff['tropical_mean']
    ],
    'Global Std (J/kg/s)': [
        stats_cntl['global_std'],
        stats_p4k['global_std'],
        stats_4co2['global_std'],
        stats_p4k_diff['global_std'],
        stats_4co2_diff['global_std']
    ]
})

print("="*80)
print("Statistical Summary of dh/dt")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

## 6. Zonal Mean Analysis

In [ ]:
# Calculate zonal means
lon_dim = 'lon' if 'lon' in dhdt_cntl_mean.dims else 'longitude'

dhdt_cntl_zonal = dhdt_cntl_mean.mean(dim=lon_dim)
dhdt_p4k_zonal = dhdt_p4k_mean.mean(dim=lon_dim)
dhdt_4co2_zonal = dhdt_4co2_mean.mean(dim=lon_dim)
dhdt_p4k_diff_zonal = dhdt_p4k_diff_mean.mean(dim=lon_dim)
dhdt_4co2_diff_zonal = dhdt_4co2_diff_mean.mean(dim=lon_dim)

# Plot zonal means
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Absolute values
lat_dim = 'lat' if 'lat' in dhdt_cntl_zonal.dims else 'latitude'
dhdt_cntl_zonal.plot(ax=ax1, label='CNTL', linewidth=2)
dhdt_p4k_zonal.plot(ax=ax1, label='P4K', linewidth=2)
dhdt_4co2_zonal.plot(ax=ax1, label='4CO2', linewidth=2)
ax1.set_xlabel('Latitude')
ax1.set_ylabel('dh/dt (J/kg/s)')
ax1.set_title('Zonal Mean dh/dt')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.5)

# Differences
dhdt_p4k_diff_zonal.plot(ax=ax2, label='P4K - CNTL', linewidth=2)
dhdt_4co2_diff_zonal.plot(ax=ax2, label='4CO2 - CNTL', linewidth=2)
ax2.set_xlabel('Latitude')
ax2.set_ylabel('Δ(dh/dt) (J/kg/s)')
ax2.set_title('Zonal Mean dh/dt Differences')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('/work/mh1498/m301257/figures/dhdt_zonal_mean.png', dpi=300, bbox_inches='tight')
plt.show()

print("Zonal mean analysis completed!")

## 7. Time Series Analysis (Tropical Average)

In [ ]:
# Calculate tropical average time series
lat_name = 'lat' if 'lat' in dhdt_cntl.dims else 'latitude'
lon_name = 'lon' if 'lon' in dhdt_cntl.dims else 'longitude'

tropical_bounds = (-15, 15)  # Tropical region definition

dhdt_cntl_tropical = dhdt_cntl.sel({lat_name: slice(tropical_bounds[0], tropical_bounds[1])})
dhdt_p4k_tropical = dhdt_p4k.sel({lat_name: slice(tropical_bounds[0], tropical_bounds[1])})
dhdt_4co2_tropical = dhdt_4co2.sel({lat_name: slice(tropical_bounds[0], tropical_bounds[1])})

dhdt_cntl_ts = dhdt_cntl_tropical.mean(dim=[lat_name, lon_name])
dhdt_p4k_ts = dhdt_p4k_tropical.mean(dim=[lat_name, lon_name])
dhdt_4co2_ts = dhdt_4co2_tropical.mean(dim=[lat_name, lon_name])

# Plot time series
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Absolute values
dhdt_cntl_ts.plot(ax=ax1, label='CNTL', alpha=0.7)
dhdt_p4k_ts.plot(ax=ax1, label='P4K', alpha=0.7)
dhdt_4co2_ts.plot(ax=ax1, label='4CO2', alpha=0.7)
ax1.set_xlabel('Time')
ax1.set_ylabel('dh/dt (J/kg/s)')
ax1.set_title(f'Tropical Average ({tropical_bounds[0]}°-{tropical_bounds[1]}°) dh/dt Time Series')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.5)

# Differences
(dhdt_p4k_ts - dhdt_cntl_ts).plot(ax=ax2, label='P4K - CNTL', alpha=0.7)
(dhdt_4co2_ts - dhdt_cntl_ts).plot(ax=ax2, label='4CO2 - CNTL', alpha=0.7)
ax2.set_xlabel('Time')
ax2.set_ylabel('Δ(dh/dt) (J/kg/s)')
ax2.set_title('Tropical Average dh/dt Differences')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('/work/mh1498/m301257/figures/dhdt_time_series.png', dpi=300, bbox_inches='tight')
plt.show()

print("Time series analysis completed!")

## 8. Save Results

In [ ]:
# Save dh/dt results to netCDF files
output_path = Path('/work/mh1498/m301257/processed_data')

# Create datasets with dh/dt
ds_dhdt_cntl = xr.Dataset({'dhdt': dhdt_cntl})
ds_dhdt_p4k = xr.Dataset({'dhdt': dhdt_p4k})
ds_dhdt_4co2 = xr.Dataset({'dhdt': dhdt_4co2})

# Add metadata
for ds, exp_name in zip([ds_dhdt_cntl, ds_dhdt_p4k, ds_dhdt_4co2], 
                         ['CNTL', 'P4K', '4CO2']):
    ds['dhdt'].attrs['long_name'] = 'Time tendency of moist static energy'
    ds['dhdt'].attrs['units'] = 'J kg-1 s-1'
    ds['dhdt'].attrs['description'] = f'dh/dt calculated from MSE data for {exp_name} experiment'

# Save to files
ds_dhdt_cntl.to_netcdf(output_path / 'dhdt_cntl.nc')
ds_dhdt_p4k.to_netcdf(output_path / 'dhdt_p4k.nc')
ds_dhdt_4co2.to_netcdf(output_path / 'dhdt_4co2.nc')

# Save difference fields
ds_dhdt_p4k_diff = xr.Dataset({'dhdt_diff': dhdt_p4k_diff})
ds_dhdt_4co2_diff = xr.Dataset({'dhdt_diff': dhdt_4co2_diff})

ds_dhdt_p4k_diff['dhdt_diff'].attrs['long_name'] = 'P4K - CNTL difference in dh/dt'
ds_dhdt_p4k_diff['dhdt_diff'].attrs['units'] = 'J kg-1 s-1'
ds_dhdt_4co2_diff['dhdt_diff'].attrs['long_name'] = '4CO2 - CNTL difference in dh/dt'
ds_dhdt_4co2_diff['dhdt_diff'].attrs['units'] = 'J kg-1 s-1'

ds_dhdt_p4k_diff.to_netcdf(output_path / 'dhdt_p4k_cntl_diff.nc')
ds_dhdt_4co2_diff.to_netcdf(output_path / 'dhdt_4co2_cntl_diff.nc')

# Save statistics to CSV
summary_df.to_csv(output_path / 'dhdt_statistics.csv', index=False)

print("="*80)
print("Results saved successfully!")
print("="*80)
print(f"dh/dt data saved to: {output_path}")
print("Files created:")
print("  - dhdt_cntl.nc")
print("  - dhdt_p4k.nc")
print("  - dhdt_4co2.nc")
print("  - dhdt_p4k_cntl_diff.nc")
print("  - dhdt_4co2_cntl_diff.nc")
print("  - dhdt_statistics.csv")
print("="*80)